<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif;">

<h1 style="text-align:center; font-size: 2.2em; margin-bottom: 8px; color: #e94560;">🧬 Algoritmo Genético para o Problema de Job-Shop Scheduling Flexível</h1>
<h2 style="text-align:center; font-size: 1.2em; color: #a8b2d8; font-weight: normal; margin-bottom: 30px;">Flexible Job-Shop Scheduling Problem (FJSP) — Instância Estática</h2>

<hr style="border-color: #e94560; margin: 20px 0;"/>

<table style="width:100%; border-collapse: collapse; color: #ccd6f6;">
  <tr>
    <td style="padding: 8px 16px; font-size: 1em;"><b>👨‍🎓 Aluno:</b></td>
    <td style="padding: 8px 16px;">Paulo Ricardo Dalsoto</td>
  </tr>
  <tr>
    <td style="padding: 8px 16px;"><b>📚 Disciplina:</b></td>
    <td style="padding: 8px 16px;">Inteligência Artificial e Aprendizado de Máquina</td>
  </tr>
  <tr>
    <td style="padding: 8px 16px;"><b>🏛️ Instituição:</b></td>
    <td style="padding: 8px 16px;">Universidade do Vale do Rio dos Sinos (UNISINOS)</td>
  </tr>
  <tr>
    <td style="padding: 8px 16px;"><b>📅 Semestre:</b></td>
    <td style="padding: 8px 16px;">2025/2</td>
  </tr>
</table>

<hr style="border-color: #e94560; margin: 20px 0;"/>

<p style="text-align:center; color: #8892b0; font-size: 0.9em;">Trabalho de implementação de algoritmo genético aplicado a problema de otimização combinatória</p>
</div>

---
## 1. Introdução

O **Problema de Job-Shop Scheduling Flexível** (*Flexible Job-Shop Scheduling Problem*, FJSP) é um dos problemas de otimização combinatória de maior relevância prática e teórica nas áreas de pesquisa operacional e inteligência artificial. Ele modela situações recorrentes em ambientes de manufatura modernos, onde um conjunto de tarefas de produção (*jobs*) precisa ser executado em um conjunto de máquinas de forma eficiente, respeitando restrições tecnológicas e operacionais.

Diferentemente do Job-Shop Scheduling clássico (JSSP), em que cada operação está rigidamente vinculada a uma única máquina, o FJSP incorpora **flexibilidade de roteamento**: cada operação pode ser processada por qualquer máquina de um subconjunto de máquinas elegíveis, cada uma com seu próprio tempo de processamento. Essa característica reflete com maior fidelidade os ambientes industriais reais, como células de manufatura flexível, centros de usinagem CNC e linhas de montagem reconfiguráveis. Por outro lado, ela torna o problema consideravelmente mais complexo, pois o algoritmo de resolução deve tomar decisões tanto de **atribuição** (em qual máquina executar cada operação) quanto de **sequenciamento** (em qual ordem as operações serão processadas em cada máquina).

O FJSP é classificado como **NP-difícil**, o que significa que não se conhece algoritmo de tempo polinomial capaz de resolvê-lo de forma ótima para instâncias de tamanho arbitrário. Por essa razão, ao longo das últimas décadas, a literatura da área desenvolveu uma ampla variedade de abordagens heurísticas e metaheurísticas, entre elas *Simulated Annealing*, Busca Tabu, Otimização por Colônia de Formigas (ACO), Otimização por Enxame de Partículas (PSO) e, em especial, os **Algoritmos Genéticos (AG)**, que têm apresentado resultados competitivos frente a outros métodos.

Neste trabalho, implementamos um Algoritmo Genético para resolver o FJSP na sua variante **estática**, em que todos os jobs e suas operações são conhecidos de antemão, antes do início do escalonamento. Essa variante se opõe ao FJSP dinâmico, no qual novos jobs chegam ao sistema ao longo do tempo. A implementação é inspirada no artigo de referência [[1]](#9-referências), que propõe e avalia um AG com codificação cromossômica dupla especialmente projetada para o FJSP, utilizando as instâncias clássicas de Brandimarte como benchmark.

O objetivo central do trabalho é minimizar o **makespan**, ou seja, o instante de conclusão do último job processado. Essa é a métrica mais amplamente adotada na literatura de scheduling e serve como indicador direto da eficiência do escalonamento obtido.

---
## 2. Formulação do FJSP Estático

### 2.1 Definição e Notação

Para entender o FJSP, é útil começar por um exemplo concreto antes de introduzir a notação formal.

Imagine uma pequena fábrica com **3 jobs** a serem produzidos e **3 máquinas** disponíveis. Cada job representa um produto diferente e precisa passar por uma série de etapas de fabricação chamadas **operações**. A ordem dessas etapas é tecnologicamente obrigatória: não é possível montar uma peça antes de usiná-la, por exemplo. No FJSP, o que diferencia esse cenário do escalonamento clássico é que cada operação pode ser realizada em mais de uma máquina, com tempos de processamento diferentes dependendo da máquina escolhida.

A figura a seguir, extraída do artigo de referência [[1]](#9-referências), ilustra exatamente esse cenário:

<img src="imgs/paper_figure1.png" width="900"/>

*Fonte: extraída do artigo de referência [[1]](#9-referências)*


<!-- ![Figura 1 do Artigo de Referência](imgs/paper_figure1.png) -->
<!-- Figura 1: Exemplo de instância FJSP com jobs, operações e máquinas elegíveis -->

Nessa figura, cada linha colorida representa um job com suas operações. Para cada operação, são listadas as máquinas que podem executá-la junto com o respectivo tempo de processamento. O algoritmo precisa decidir, para cada operação, qual máquina será usada e em que momento ela será iniciada, sempre respeitando a ordem das operações dentro de cada job e garantindo que nenhuma máquina processe duas operações ao mesmo tempo.

**Notação utilizada no restante do trabalho:**

Formalizando o que foi descrito acima: o problema envolve um conjunto de $n$ jobs processados em $m$ máquinas. Cada job $J_i$ é composto por uma sequência de $n_i$ operações que devem ser executadas nessa ordem. Para cada operação $O_{i,j}$ (a $j$-ésima operação do job $i$), existe um subconjunto $M_{i,j}$ de máquinas elegíveis; caso a operação seja atribuída à máquina $k$, ela requer $p_{i,j,k}$ unidades de tempo para ser concluída. Todos esses dados são conhecidos de antemão, o que define a variante **estática** do problema.

O algoritmo precisa determinar duas coisas para cada operação: em qual máquina ela será executada (decisão de **atribuição**) e em que instante ela será iniciada (decisão de **escalonamento**). Uma solução válida é qualquer combinação dessas decisões que respeite todas as restrições descritas na seção seguinte.

### 2.2 Restrições

Toda solução válida para o FJSP precisa respeitar quatro tipos de restrição. A seguir, cada uma é apresentada com uma breve explicação e um exemplo baseado nas instâncias de Brandimarte.

**R1 - Atribuição única**

Cada operação deve ser atribuída a **exatamente uma** das máquinas do seu conjunto elegível. O algoritmo não pode deixar uma operação sem máquina, nem dividi-la entre duas máquinas.

*Exemplo:* $O_{1,1}$ do Job 1 pode ir a $M_1$ (tempo=5) ou $M_3$ (tempo=4). O algoritmo escolhe uma delas. Uma solução que não faça essa escolha, ou que atribua $O_{1,1}$ a $M_2$ (que não é elegível), é inválida.

$$a_{i,j} \in M_{i,j}, \quad \forall\, i, j$$

**R2 - Precedência (ordem tecnológica)**

As operações de um job precisam ser executadas **na ordem em que aparecem**. A próxima operação só pode começar depois que a anterior terminar completamente.

*Exemplo:* No Job 1, a operação $O_{1,2}$ não pode começar antes de $O_{1,1}$ ser concluída. Se $O_{1,1}$ começa no instante 0 e dura 4 unidades de tempo (caso seja alocada em $M_3$), então $O_{1,2}$ só pode começar no instante 4 ou depois.

$$S_{i,j} + p_{i,j,a_{i,j}} \leq S_{i,j+1}$$

**R3 - Capacidade de máquina (sem sobreposição)**

Cada máquina só pode processar **uma operação por vez**. Duas operações atribuídas à mesma máquina não podem se sobrepor no tempo.

*Exemplo:* Suponha que $O_{1,3}$ e $O_{2,1}$ (de jobs diferentes) sejam ambas atribuídas a $M_3$. Se $O_{1,3}$ começa no instante 5 e dura 4 unidades, então $O_{2,1}$ só pode começar no instante 9 ou mais tarde — ou terminar antes do instante 5. Não existe a opção de ambas rodarem ao mesmo tempo.

$$S_{i,j} + p_{i,j,k} \leq S_{i',j'} \quad \text{ou} \quad S_{i',j'} + p_{i',j',k} \leq S_{i,j}$$

**R4 - Não-preempção**

Uma vez que uma operação começa a ser processada, ela não pode ser pausada ou interrompida para dar lugar a outra. O processamento é sempre contínuo do início ao fim.

*Exemplo:* Se $O_{1,4}$ foi atribuída a $M_1$ com tempo de processamento igual a 1, ela ocupa $M_1$ por exatamente 1 unidade de tempo, sem interrupção.

### 2.3 Função Objetivo

O objetivo é minimizar o **makespan** $C_{\max}$, definido como o instante de conclusão da última operação entre todos os jobs:

$$\text{Minimizar} \quad C_{\max} = \max_{i=1}^{n} C_i = \max_{i=1}^{n} \left( S_{i,n_i} + p_{i,n_i,\, a_{i,n_i}} \right)$$

onde $C_i = S_{i,n_i} + p_{i,n_i,\, a_{i,n_i}}$ é o tempo de conclusão do job $J_i$.

O makespan representa o tempo total de ocupação do sistema de produção, sendo um indicador direto da eficiência do escalonamento.

### 2.4 Complexidade e Espaço de Busca

O FJSP é **NP-difícil** mesmo em versões simplificadas. Isso decorre da combinação de dois subproblemas, ambos NP-difíceis de forma independente:

| Subproblema | Descrição | Denominação |
|-------------|----------|-------------|
| *Routing* | Decidir qual máquina executa cada operação | Atribuição |
| *Scheduling* | Decidir a ordem das operações em cada máquina | Sequenciamento |

O número de soluções candidatas cresce de forma super-exponencial com o tamanho do problema. Para as instâncias utilizadas neste trabalho:

| Instância | $n$ | $m$ | Operações | Flex. média | Espaço de busca (ordem de magnitude) |
|-----------|-----|-----|-----------|-------------|--------------------------------------|
| Mk1 | 10 | 6 | 55 | 2.09 | $\sim 10^{57}$ |
| Mk2 | 10 | 6 | 58 | 4.10 | $\sim 10^{60}$ |
| Mk3 | 15 | 8 | 150 | 3.01 | $\sim 10^{130}$ |

Para efeito de comparação, o número estimado de átomos no universo observável é $\sim 10^{80}$. Essa dimensionalidade torna inviável a busca exaustiva e justifica o uso de metaheurísticas como os **Algoritmos Genéticos**.

> **Observação sobre a variante estática:** No FJSP estático, o conjunto completo de jobs, suas operações e os respectivos tempos de processamento são conhecidos antes do início do escalonamento. Isso contrasta com o FJSP dinâmico, no qual novos jobs chegam ao sistema ao longo do tempo, exigindo reescalonamento adaptativo. Neste trabalho, tratamos exclusivamente da variante **estática**.

---
## 3. Referência Bibliográfica Base

A implementação e a metodologia adotadas neste trabalho são inspiradas no seguinte artigo:

> **[1]** *A Genetic Algorithm for the Flexible Job-Shop Scheduling Problem.*  
> ACM Digital Library, 2024.  
> DOI: [10.1145/3698300.3698316](https://dl.acm.org/doi/10.1145/3698300.3698316)

O artigo propõe um Algoritmo Genético com **representação cromossômica dupla** para lidar simultaneamente com os dois subproblemas do FJSP (atribuição e sequenciamento). Os principais aspectos abordados incluem:

- **Codificação dos indivíduos**: cromossomo bipartido, em que uma parte codifica a atribuição de máquinas (*machine assignment*) e outra codifica a ordem de execução das operações (*operation sequence*);
- **Inicialização da população**: uso de heurísticas construtivas (como atribuição à máquina de menor tempo de processamento) combinadas com indivíduos aleatórios;
- **Operadores genéticos**: operadores de cruzamento e mutação adaptados à estrutura dupla do cromossomo;
- **Avaliação experimental**: benchmarks de Brandimarte (Mk1 a Mk10), com comparação frente a outros algoritmos da literatura.

O artigo é utilizado neste trabalho como **referência de comparação**: os resultados obtidos pelo algoritmo implementado serão avaliados frente aos valores reportados no artigo para as mesmas instâncias de Brandimarte. A implementação do Algoritmo Genético em si é de autoria própria, desenvolvida de forma independente sem embasamento direto na implementação descrita no artigo.

---
## 4. Instâncias Utilizadas (Brandimarte)

### 4.1 O Conjunto de Benchmarks de Brandimarte

As instâncias utilizadas neste trabalho pertencem ao conjunto de benchmarks proposto por **Brandimarte (1993)** [[2]](#9-referências), que se tornou uma das referências mais utilizadas na literatura do FJSP. O conjunto é composto por 10 instâncias (Mk1 a Mk10), caracterizadas por diferentes tamanhos e graus de flexibilidade.

Neste trabalho, utilizamos as três primeiras instâncias:

| Instância | Jobs ($n$) | Máquinas ($m$) | Operações | Flex. Média | Melhor Makespan Conhecido |
|-----------|-----------|---------------|-----------|-------------|---------------------------|
| **Mk1** | 10 | 6 | 55 | 2.09 | 40 |
| **Mk2** | 10 | 6 | 58 | 4.10 | 26 |
| **Mk3** | 15 | 8 | 150 | 3.01 | 204 |

A **flexibilidade média** indica, em média, quantas máquinas são elegíveis por operação. Quanto maior esse valor, mais amplo o subproblema de atribuição e maior a dificuldade combinatória.

### 4.2 Formato do Arquivo `.fjs`

As instâncias são armazenadas no formato padrão `.fjs`. Sua estrutura é:

```
Linha 1 :  <nJobs>  <nMáquinas>  [<flexibilidadeMedia>]

Linhas seguintes (uma por job):
  <nOps>  <nMaq_1> <maq> <tempo> ... <nMaq_2> <maq> <tempo> ...  ...
  └──┬──┘  └────────────┬────────┘   └────────────┬────────┘
  nº ops    operação 1              operação 2
```

Cada operação é descrita por: o número de máquinas elegíveis, seguido dos pares `(índice_máquina, tempo_processamento)`.

### 4.3 Exemplo Real — BrandimarteMk1.fjs, Job 1

O arquivo `BrandimarteMk1.fjs` começa com:

```
10  6  2
 6  2 1 5 3 4  3 5 3 3 5 2 1  2 3 4 6 2  3 6 5 2 6 1 1  1 3 1  3 6 6 3 6 4 3
```

**Cabeçalho:** 10 jobs, 6 máquinas, flexibilidade média = 2.

**Job 1** possui `6` operações. Decodificando cada operação:

| Operação | Dado bruto | Máquinas elegíveis ($M_{1,j}$) | Tempos ($p_{1,j,k}$) |
|----------|-----------|-------------------------------|----------------------|
| $O_{1,1}$ | `2  1 5  3 4` | $\{M_1, M_3\}$ | $p_{1,1,1}=5,\ p_{1,1,3}=4$ |
| $O_{1,2}$ | `3  5 3  3 5  2 1` | $\{M_5, M_3, M_2\}$ | $p_{1,2,5}=3,\ p_{1,2,3}=5,\ p_{1,2,2}=1$ |
| $O_{1,3}$ | `2  3 4  6 2` | $\{M_3, M_6\}$ | $p_{1,3,3}=4,\ p_{1,3,6}=2$ |
| $O_{1,4}$ | `3  6 5  2 6  1 1` | $\{M_6, M_2, M_1\}$ | $p_{1,4,6}=5,\ p_{1,4,2}=6,\ p_{1,4,1}=1$ |
| $O_{1,5}$ | `1  3 1` | $\{M_3\}$ | $p_{1,5,3}=1$ |
| $O_{1,6}$ | `3  6 6  3 6  4 3` | $\{M_6, M_3, M_4\}$ | $p_{1,6,6}=6,\ p_{1,6,3}=6,\ p_{1,6,4}=3$ |

Isso evidencia o caráter **FJSP** da instância: 5 das 6 operações do Job 1 possuem 2 ou 3 máquinas elegíveis. A atribuição de máquinas, portanto, não é determinada pelo problema; ela é uma **variável de decisão** que o algoritmo deve otimizar.

Apenas $O_{1,5}$ tem máquina fixa ($M_3$), comportando-se como uma operação de JSSP clássico. Esse misto de operações fixas e flexíveis é típico do **FJSP Parcial** (*Partial Flexibility*), categoria à qual pertencem as instâncias de Brandimarte.

#### Tabela de Resultados do Artigo de Referência

A figura a seguir apresenta os resultados reportados pelo artigo base, comparando o desempenho do AG proposto com outros métodos da literatura nas instâncias de Brandimarte:

<img src="imgs/paper_table2.png" width="900"/>

*Fonte: extraída do artigo de referência [[1]](#9-referências)*

---
## 5. Modelagem para o Algoritmo Genético

> 🔧 *Esta seção será desenvolvida na próxima etapa do trabalho.*

Tópicos a cobrir:
- Representação cromossômica (codificação dupla: atribuição + sequenciamento)
- Decodificação do cromossomo para solução (cálculo do makespan via *schedule*)
- Função de aptidão (*fitness*)
- Inicialização da população

---
## 6. Implementação

> 🔧 *Esta seção será desenvolvida na próxima etapa do trabalho.*

Tópicos a cobrir:
- Seleção (torneio, roleta)
- Cruzamento (*crossover*)
- Mutação
- Parâmetros do AG (tamanho da população, taxa de cruzamento, taxa de mutação, critério de parada)
- Código-fonte

---
## 7. Experimentos e Resultados

> 🔧 *Esta seção será desenvolvida na próxima etapa do trabalho.*

Tópicos a cobrir:
- Configuração experimental (hardware, repetições, sementes)
- Resultados por instância (Mk1, Mk2, Mk3): melhor, médio, pior makespan
- Comparação com os valores do artigo de referência
- Curva de convergência do AG

---
## 8. Conclusão

> 🔧 *Esta seção será desenvolvida ao final do trabalho.*

---
## 9. Referências

**[1]** *A Genetic Algorithm for the Flexible Job-Shop Scheduling Problem.* ACM Digital Library, 2024.  
DOI: [10.1145/3698300.3698316](https://dl.acm.org/doi/10.1145/3698300.3698316)

**[2]** Brandimarte, P. (1993). Routing and scheduling in a flexible job shop by tabu search. *Annals of Operations Research*, 41(3), 157–183.

**[3]** Kacem, I., Hammadi, S., & Borne, P. (2002). Approach by localization and multiobjective evolutionary optimization for flexible job-shop scheduling problems. *IEEE Transactions on Systems, Man, and Cybernetics*, 32(1), 408–419.

**[4]** Xia, W., & Wu, Z. (2005). An effective hybrid optimization approach for multi-objective flexible job-shop scheduling problems. *Computers & Industrial Engineering*, 48(2), 409–425.

**[5]** Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness.* W. H. Freeman.